In [13]:
import sys
print(sys.executable)

from Bio.Seq import Seq
from Bio.Data import CodonTable



c:\Python314\python.exe


In [14]:
def validate_dna(sequence: str) -> str:
    """Validate a DNA sequence: uppercase, valid bases, length divisible by 3."""
    seq = sequence.strip().upper()
    valid_bases = set("ATGC")
    if not set(seq).issubset(valid_bases):
        raise ValueError(f"Invalid characters found in sequence: {set(seq) - valid_bases}")
    if len(seq) % 3 != 0:
        raise ValueError(f"Sequence length ({len(seq)}) is not divisible by 3")
    return seq



In [15]:
def translate_dna(sequence: str) -> str:
    """Translate a validated DNA sequence into its protein sequence."""
    seq = validate_dna(sequence)
    return str(Seq(seq).translate(to_stop=True))



In [16]:
def split_into_codons(sequence: str) -> list[str]:
    """Split a validated DNA sequence into a list of codons."""
    seq = validate_dna(sequence)
    return [seq[i:i+3] for i in range(0, len(seq), 3)]


In [17]:
def gc_content(sequence: str) -> float:
    """Return GC content as a percentage."""
    seq = sequence.strip().upper()
    g = seq.count("G")
    c = seq.count("C")
    return (g + c) / len(seq) * 100


In [18]:
test_seq = "ATGGCTAAATAA"  # Met-Ala-Lys-Stop
print("Codons:", split_into_codons(test_seq))
print("Protein:", translate_dna(test_seq))

Codons: ['ATG', 'GCT', 'AAA', 'TAA']
Protein: MAK


In [19]:
CODON_USAGE_ECOLI = {
    "F": {"TTT": 0.57, "TTC": 0.43},
    "L": {"TTA": 0.15, "TTG": 0.12, "CTT": 0.12, "CTC": 0.10, "CTA": 0.05, "CTG": 0.46},
    "I": {"ATT": 0.58, "ATC": 0.35, "ATA": 0.07},
    "M": {"ATG": 1.00},
    "V": {"GTT": 0.25, "GTC": 0.18, "GTA": 0.17, "GTG": 0.40},
    "S": {"TCT": 0.11, "TCC": 0.11, "TCA": 0.15, "TCG": 0.16, "AGT": 0.14, "AGC": 0.33},
    "P": {"CCT": 0.17, "CCC": 0.13, "CCA": 0.14, "CCG": 0.55},
    "T": {"ACT": 0.16, "ACC": 0.47, "ACA": 0.13, "ACG": 0.24},
    "A": {"GCT": 0.11, "GCC": 0.31, "GCA": 0.21, "GCG": 0.38},
    "Y": {"TAT": 0.53, "TAC": 0.47},
    "H": {"CAT": 0.55, "CAC": 0.45},
    "Q": {"CAA": 0.30, "CAG": 0.70},
    "N": {"AAT": 0.47, "AAC": 0.53},
    "K": {"AAA": 0.73, "AAG": 0.27},
    "D": {"GAT": 0.65, "GAC": 0.35},
    "E": {"GAA": 0.70, "GAG": 0.30},
    "C": {"TGT": 0.42, "TGC": 0.58},
    "W": {"TGG": 1.00},
    "R": {"CGT": 0.36, "CGC": 0.44, "CGA": 0.07, "CGG": 0.07, "AGA": 0.02, "AGG": 0.03},
    "G": {"GGT": 0.29, "GGC": 0.46, "GGA": 0.13, "GGG": 0.12},
    "*": {"TAA": 0.64, "TAG": 0.00, "TGA": 0.36},
}

In [20]:

def best_codon(amino_acid: str) -> str:
    """Return the single highest-frequency codon for an amino acid in E. coli."""
    codons = CODON_USAGE_ECOLI[amino_acid]
    return max(codons, key=codons.get)
 
 
def ranked_codons(amino_acid: str) -> list[str]:
    """Return synonymous codons for an amino acid, ranked highest-frequency first."""
    codons = CODON_USAGE_ECOLI[amino_acid]
    return sorted(codons, key=codons.get, reverse=True)
 

In [21]:
print("Best codon for Leucine (L):", best_codon("L"))
print("Ranked codons for Arginine (R):", ranked_codons("R"))
print("Best codon for Methionine (M):", best_codon("M"))

Best codon for Leucine (L): CTG
Ranked codons for Arginine (R): ['CGC', 'CGT', 'CGA', 'CGG', 'AGG', 'AGA']
Best codon for Methionine (M): ATG


In [22]:
def verify_translation(original_dna: str, optimized_dna: str) -> bool:
    """Confirm that optimization preserved the original protein sequence."""
    original_protein = translate_dna(original_dna)
    optimized_protein = translate_dna(optimized_dna)
    if original_protein != optimized_protein:
        raise ValueError(
            f"Protein mismatch!\nOriginal:  {original_protein}\nOptimized: {optimized_protein}"
        )
    return True

In [24]:


import math

CODON_TO_AMINO_ACID = {
    codon: aa
    for aa, codon_dict in CODON_USAGE_ECOLI.items()
    for codon in codon_dict
}

def _build_relative_adaptiveness():
    weights = {}
    for aa, codon_dict in CODON_USAGE_ECOLI.items():
        if aa == "*":
            continue
        max_freq = max(codon_dict.values())
        for codon, freq in codon_dict.items():
            weights[codon] = freq / max_freq
    return weights

RELATIVE_ADAPTIVENESS = _build_relative_adaptiveness()

def calculate_cai(dna_sequence: str) -> float:
    """Calculate CAI for a DNA sequence relative to E. coli K12 codon usage."""
    codons = split_into_codons(dna_sequence)
    w_values = [
        RELATIVE_ADAPTIVENESS[codon]
        for codon in codons
        if codon in RELATIVE_ADAPTIVENESS
    ]
    if not w_values:
        raise ValueError("No valid (non-stop) codons found to calculate CAI")
    log_sum = sum(math.log(w) for w in w_values)
    return math.exp(log_sum / len(w_values))


CAI measures how closely a sequence's codon usage matches the preferred
codon usage of a reference host (here, E. coli K12).

Formula:
    For each codon, its "relative adaptiveness" (w) is:
        w = frequency of this codon / frequency of the most-used synonymous codon
    (so the most-preferred codon for each amino acid always has w = 1.0)

    CAI of a sequence = geometric mean of w values across all codons in the sequence
                       = (w1 * w2 * ... * wn) ** (1/n)

CAI ranges from 0 to 1. Higher = closer to the host's preferred codon usage.
Stop codons are excluded from the calculation (standard convention).

In [25]:

def compare_sequences(original_dna: str, optimized_dna: str) -> dict:
    """
    Build a full comparison report between an original and optimized DNA sequence.
    """

    original_protein = translate_dna(original_dna)
    optimized_protein = translate_dna(optimized_dna)


    verify_translation(original_dna, optimized_dna)


    original_codons = split_into_codons(original_dna)
    optimized_codons = split_into_codons(optimized_dna)

    changed_positions = [
        i for i, (orig, opt) in enumerate(zip(original_codons, optimized_codons))
        if orig != opt
    ]

    percent_changed = (len(changed_positions) / len(original_codons)) * 100

    report = {
        "original_dna": original_dna,
        "optimized_dna": optimized_dna,
        "protein": original_protein,  # same for both, since verification passed
        "cai_original": calculate_cai(original_dna),
        "cai_optimized": calculate_cai(optimized_dna),
        "gc_original": gc_content(original_dna),
        "gc_optimized": gc_content(optimized_dna),
        "num_codons": len(original_codons),
        "num_codons_changed": len(changed_positions),
        "percent_codons_changed": percent_changed,
        "changed_positions": changed_positions,
    }
    return report


def print_report(report: dict) -> None:
    """Pretty-print a comparison report."""
    print("=" * 50)
    print("CODON OPTIMIZATION REPORT")
    print("=" * 50)
    print(f"Protein sequence:      {report['protein']}")
    print(f"Original DNA:          {report['original_dna']}")
    print(f"Optimized DNA:         {report['optimized_dna']}")
    print("-" * 50)
    print(f"CAI (original):        {report['cai_original']:.4f}")
    print(f"CAI (optimized):       {report['cai_optimized']:.4f}")
    print(f"GC% (original):        {report['gc_original']:.2f}%")
    print(f"GC% (optimized):       {report['gc_optimized']:.2f}%")
    print("-" * 50)
    print(f"Total codons:          {report['num_codons']}")
    print(f"Codons changed:        {report['num_codons_changed']} "
          f"({report['percent_codons_changed']:.1f}%)")
    print(f"Changed positions:     {report['changed_positions']}")
    print("=" * 50)


def optimize_sequence(dna_sequence: str) -> str:
    """
    Given a validated DNA sequence, return a new DNA sequence optimized
    for E. coli codon usage, encoding the exact same protein.
    """
    protein = translate_dna(dna_sequence)
    optimized_codons = [best_codon(aa) for aa in protein]
    optimized_codons.append("TAA")  # re-append stop codon (translate_dna strips it)
    return "".join(optimized_codons)

original_seq = "ATGGCTAAATAA"
optimized_seq = optimize_sequence(original_seq)

report = compare_sequences(original_seq, optimized_seq)
print_report(report)

CODON OPTIMIZATION REPORT
Protein sequence:      MAK
Original DNA:          ATGGCTAAATAA
Optimized DNA:         ATGGCGAAATAA
--------------------------------------------------
CAI (original):        0.6615
CAI (optimized):       1.0000
GC% (original):        25.00%
GC% (optimized):       33.33%
--------------------------------------------------
Total codons:          4
Codons changed:        1 (25.0%)
Changed positions:     [1]


Variable             Type        Data/Info
------------------------------------------
best_codon           function    <function best_codon at 0x0000029907690510>
compare_sequences    function    <function compare_sequenc<...>es at 0x00000299076907D0>
gc_content           function    <function gc_content at 0x0000029907690250>
print_report         function    <function print_report at 0x0000029907690930>
ranked_codons        function    <function ranked_codons at 0x0000029907690670>
split_into_codons    function    <function split_into_codo<...>ns at 0x000002990768BED0>
translate_dna        function    <function translate_dna at 0x00000299076900F0>
validate_dna         function    <function validate_dna at 0x000002990768BE20>
verify_translation   function    <function verify_translat<...>on at 0x00000299076903B0>
